# Zero-Shot Annotation on Kaggle GPU

Runs the Grounding DINO annotation stage of `collect_and_annotate.py` on a
Kaggle T4 and writes a validated COCO file.

## Setup (once)

1. Create a Kaggle Dataset containing just two small files:
   - `collect_and_annotate.py`
   - `manifest.json`  (from `data/manifest.json`)
2. Attach it to this notebook via **Add Data**.
3. Settings: **Accelerator = GPU T4 x2**, **Internet = On**.

The images are *not* uploaded. The manifest records every source URL, so
this notebook re-downloads them directly from Wikimedia Commons. That keeps
the upload at a few hundred KB instead of a couple of GB, and it is also a
live test that the manifest really is reproducible.

## Why one GPU, not two

Detection is roughly 0.2 s/image on a single T4, so the whole dataset takes
about two minutes. Sharding across both GPUs would add process management
and result-merging for no meaningful saving, so this notebook keeps to one.
The second GPU is worth using later, for Phase 2 training.

## 1. Environment check

In [ ]:
import sys

import torch

print(f"torch {torch.__version__}  cuda={torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(i)
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  GPU {i}: {name}  {total:.1f} GB")

assert torch.cuda.is_available(), "Enable the GPU accelerator in Settings."

In [ ]:
# Kaggle ships torch and transformers, but pin the pieces the script needs.
!pip install -q transformers==4.46.3 pycocotools==2.0.8 2>&1 | tail -2

## 2. Stage the project files

`collect_and_annotate.py` resolves its data paths relative to its own
location, so copying it into `/kaggle/working` puts the dataset at
`/kaggle/working/data/` with no code changes.

In [ ]:
import shutil
from pathlib import Path

WORK = Path("/kaggle/working")
DATA = WORK / "data"
IMAGES = DATA / "images"
IMAGES.mkdir(parents=True, exist_ok=True)


def find_input(filename: str) -> Path:
    """Locate a file anywhere under /kaggle/input."""
    hits = sorted(Path("/kaggle/input").rglob(filename))
    if not hits:
        raise FileNotFoundError(
            f"{filename} not found under /kaggle/input. "
            "Attach the dataset containing it via 'Add Data'."
        )
    return hits[0]


shutil.copy(find_input("collect_and_annotate.py"), WORK / "collect_and_annotate.py")
shutil.copy(find_input("manifest.json"), DATA / "manifest.json")

sys.path.insert(0, str(WORK))
print("staged:", [p.name for p in WORK.iterdir() if p.is_file()])

## 3. Re-download the images from the manifest

Wikimedia throttles generic clients, so the project's descriptive
`USER_AGENT` is reused here. Downloads run in a small thread pool: enough to
be quick, few enough to stay polite. Files already present are skipped, so
this cell is safe to re-run after a session restart.

In [ ]:
import json
from concurrent.futures import ThreadPoolExecutor

import requests
from collect_and_annotate import USER_AGENT, sha256_of

manifest = json.loads((DATA / "manifest.json").read_text(encoding="utf-8"))
accepted = [r for r in manifest["images"] if r["status"] == "accepted"]
print(f"{len(accepted)} accepted images in the manifest")


def fetch(record: dict) -> tuple[str, str]:
    """Download one image and confirm it matches the recorded hash."""
    dest = IMAGES / record["filename"]
    if dest.exists():
        return record["filename"], "cached"
    try:
        resp = requests.get(
            record["image_url"],
            headers={"User-Agent": USER_AGENT, "Accept": "image/*"},
            timeout=60,
        )
        resp.raise_for_status()
        dest.write_bytes(resp.content)
    except Exception as exc:
        return record["filename"], f"failed: {type(exc).__name__}"
    # The manifest's hash is the reproducibility guarantee; verify it.
    if sha256_of(dest) != record["sha256"]:
        dest.unlink(missing_ok=True)
        return record["filename"], "hash mismatch"
    return record["filename"], "ok"


with ThreadPoolExecutor(max_workers=6) as pool:
    results = list(pool.map(fetch, accepted))

from collections import Counter

print(Counter(status for _, status in results))
for name, status in results:
    if status not in {"ok", "cached"}:
        print(f"  {status}: {name}")

## 4. Annotate on the GPU

`--shortest-edge 800` is Grounding DINO's native training resolution. On CPU
that setting costs ~75 s/image, which is why it was reduced locally; on a T4
there is no reason to compromise, so full resolution is used here.

In [ ]:
%cd /kaggle/working
!python -u collect_and_annotate.py annotate \
    --device cuda \
    --shortest-edge 800 \
    --box-threshold 0.30 \
    --text-threshold 0.25

## 5. Dataset statistics

These numbers go straight into the README, and they are also the ground
truth the Phase 3 auditor will later be checked against.

In [ ]:
import statistics

from collect_and_annotate import COCO_PATH, validate_coco

summary = validate_coco(COCO_PATH)
print(json.dumps(summary, indent=2))

coco = json.loads(COCO_PATH.read_text(encoding="utf-8"))
names = {c["id"]: c["name"] for c in coco["categories"]}
areas = [a["area"] for a in coco["annotations"]]
per_image = Counter(a["image_id"] for a in coco["annotations"])

print(f"\nboxes/image: mean {statistics.mean(per_image.values()):.1f}, "
      f"median {statistics.median(per_image.values()):.0f}")
print(f"box area px^2: min {min(areas):.0f}, median {statistics.median(areas):.0f}, "
      f"max {max(areas):.0f}")

counts = summary["per_class"]
biggest, smallest = max(counts.values()), max(1, min(counts.values()))
print(f"class imbalance ratio: {biggest / smallest:.1f}:1")
for name, count in sorted(counts.items(), key=lambda kv: -kv[1]):
    share = 100 * count / max(1, sum(counts.values()))
    print(f"  {name:<14} {count:>6}  {share:5.1f}%")

## 6. Visual spot-check

The single most useful artefact for the README. Automated labels cannot be
trusted on numbers alone, so draw them and look.

In [ ]:
import random

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

random.seed(0)
by_image = {}
for ann in coco["annotations"]:
    by_image.setdefault(ann["image_id"], []).append(ann)

palette = {"car": "#e63946", "truck": "#f4a261",
           "traffic light": "#2a9d8f", "person": "#457b9d"}
sample = random.sample([i for i in coco["images"] if by_image.get(i["id"])],
                       k=min(9, len(by_image)))

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
for ax, info in zip(axes.flat, sample):
    image = Image.open(IMAGES / info["file_name"]).convert("RGB")
    scale = 900 / max(image.size)
    image = image.resize((int(image.width * scale), int(image.height * scale)))
    ax.imshow(image)
    for ann in by_image[info["id"]]:
        x, y, w, h = [v * scale for v in ann["bbox"]]
        label = names[ann["category_id"]]
        ax.add_patch(patches.Rectangle((x, y), w, h, fill=False,
                                       edgecolor=palette[label], linewidth=1.6))
        ax.text(x, max(y - 4, 8), f"{label} {ann['score']:.2f}",
                fontsize=7, color="white",
                bbox=dict(facecolor=palette[label], edgecolor="none", pad=1))
    ax.set_title(f"id {info['id']} — {len(by_image[info['id']])} boxes", fontsize=9)
    ax.axis("off")

for ax in axes.flat[len(sample):]:
    ax.axis("off")
plt.tight_layout()
plt.savefig(WORK / "annotation_samples.png", dpi=110, bbox_inches="tight")
plt.show()

## 7. Known annotation-noise check

Grounding DINO emits overlapping boxes on crowds. Quantify that now rather
than discovering it during training — these counts become real findings for
the Phase 3 auditor instead of invented ones.

In [ ]:
def iou(a: list[float], b: list[float]) -> float:
    ax, ay, aw, ah = a
    bx, by, bw, bh = b
    x1, y1 = max(ax, bx), max(ay, by)
    x2, y2 = min(ax + aw, bx + bw), min(ay + ah, by + bh)
    overlap = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    union = aw * ah + bw * bh - overlap
    return overlap / union if union > 0 else 0.0


duplicates = 0
for anns in by_image.values():
    for i in range(len(anns)):
        for j in range(i + 1, len(anns)):
            if (anns[i]["category_id"] == anns[j]["category_id"]
                    and iou(anns[i]["bbox"], anns[j]["bbox"]) > 0.7):
                duplicates += 1

total = len(coco["annotations"])
tiny = sum(1 for a in coco["annotations"] if a["area"] < 32 * 32)
print(f"near-duplicate pairs (IoU>0.7, same class): {duplicates}")
print(f"boxes under 32x32 px: {tiny}  ({100 * tiny / total:.1f}% of {total})")
print("\nNote: these survive into the dataset on purpose. Phase 3's auditor")
print("is meant to find them; filtering them here would hide the problem.")

## 8. Download the results

Commit the notebook (or use the Output panel) and pull `annotations.json`
plus the sample grid back into the repo under `data/`.

In [ ]:
shutil.copy(COCO_PATH, WORK / "annotations.json")
size_kb = (WORK / "annotations.json").stat().st_size / 1024
print(f"/kaggle/working/annotations.json      {size_kb:,.0f} KB")
print(f"/kaggle/working/annotation_samples.png")
print("\nBoth appear in the notebook's Output panel for download.")